# LP Example 2: Transportation Problem

This example solves a **linear programming (LP)** transportation problem using OptArrow.

## Problem Description

A company has two factories (supply nodes) and three warehouses (demand nodes).
We want to minimize the total shipping cost.

**Supplies:** Factory 1 = 120 units, Factory 2 = 80 units  
**Demands:** Warehouse 1 = 70 units, Warehouse 2 = 90 units, Warehouse 3 = 40 units  
**Unit costs:**
```
         W1   W2   W3
Factory1  2    3    1
Factory2  5    4    6
```

Formulated as a standard LP with 6 decision variables (shipment amounts) and 5 equality constraints (supply/demand balance).

## 1. Set up the LP

Variables: x[i,j] = units shipped from factory i to warehouse j  
Minimize: 2x11 + 3x12 + x13 + 5x21 + 4x22 + 6x23

In [ ]:
import pyarrow as pa
import requests

# Decision variables: [x11, x12, x13, x21, x22, x23]
# Constraints (equality):
#   Supply Factory1: x11 + x12 + x13 = 120
#   Supply Factory2: x21 + x22 + x23 = 80
#   Demand W1: x11 + x21 = 70
#   Demand W2: x12 + x22 = 90
#   Demand W3: x13 + x23 = 40

ipc_dict = {
    "model": {
        "A": {
            "row": [0, 0, 0, 1, 1, 1, 2, 2, 3, 3, 4, 4],
            "col": [0, 1, 2, 3, 4, 5, 0, 3, 1, 4, 2, 5],
            "val": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0,
                    1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
        },
        "b": [120.0, 80.0, 70.0, 90.0, 40.0],
        "c": [2.0, 3.0, 1.0, 5.0, 4.0, 6.0],
        "lb": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "ub": [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0],
        "csense": ["E", "E", "E", "E", "E"],
        "osense": "min"
    },
    "model_name": "transportation_lp",
    "engine": "julia",
    "solver": {
        "solver_name": "HiGHS",
        "solver_type": "LP",
        "solver_params": {}
    }
}

## 2. Send to OptArrow and solve

In [ ]:
pa_arrays = [pa.array([v]) for v in ipc_dict.values()]
table = pa.Table.from_arrays(pa_arrays, names=list(ipc_dict.keys()))

sink = pa.BufferOutputStream()
with pa.ipc.new_stream(sink, table.schema) as writer:
    writer.write(table)
ipc_bytes = sink.getvalue().to_pybytes()

headers = {"Content-Type": "application/vnd.apache.arrow.stream"}
response = requests.post("http://localhost:8000/compute", data=ipc_bytes, headers=headers)

reader = pa.ipc.open_stream(response.content)
result_table = reader.read_all()
result = {name: result_table.column(name).to_pylist() for name in result_table.column_names}
print(result)

## 3. Interpret results

Expected optimal cost: **490** (x11=70, x12=10, x13=40, x21=0, x22=80, x23=0)

In [ ]:
if result.get('success')[0]:
    shipments = result['solution'][0]
    labels = ['F1→W1', 'F1→W2', 'F1→W3', 'F2→W1', 'F2→W2', 'F2→W3']
    for label, val in zip(labels, shipments):
        print(f"{label}: {val:.1f} units")
    print(f"\nTotal cost: {result['obj_val'][0]:.2f}")